# Event Mapping Analysis

This notebook investigates the candidate emotional-drift episodes identified in the previous emotional-drift analysis.

The objective is to determine whether each detected transition is more consistent with:

- a genuine change in review emotions,
- a change in rating patterns,
- attraction-composition effects,
- small-sample effects,
- review-text themes,
- or a temporally corresponding external event.

The analysis is performed before making any causal interpretation.

The candidate set contains:

- 2 Extreme Drift episodes: Robust Z > 3.5
- 10 Notable Drift episodes: 2.5 < Robust Z <= 3.5
- 12 candidate transitions across 8 cities

In [51]:
# ============================================================
# STEP 1: SETUP AND FILE PATHS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Project folder
project_dir = Path(
    r"C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews"
)

# Results folder
results_dir = project_dir / "results"

# Input files
reviews_file = project_dir / "Tourism_Reviews_Emotion.csv"
candidates_file = results_dir / "Event_Mapping_Candidates.csv"

print("Reviews file:")
print(reviews_file)

print("\nCandidate events file:")
print(candidates_file)

Reviews file:
C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews\Tourism_Reviews_Emotion.csv

Candidate events file:
C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews\results\Event_Mapping_Candidates.csv


In [21]:
# ============================================================
# STEP 2: LOAD REVIEW DATA AND EVENT-MAPPING CANDIDATES
# ============================================================

reviews = pd.read_csv(reviews_file)
candidates = pd.read_csv(candidates_file)

print("Review dataset shape:", reviews.shape)
print("Candidate table shape:", candidates.shape)

print("\nReview columns:")
print(reviews.columns.tolist())

print("\nCandidate columns:")
print(candidates.columns.tolist())

Review dataset shape: (22092, 30)
Candidate table shape: (12, 19)

Review columns:
['City', 'Place', 'Page', 'Review_Text', 'Review_Date', 'Reviewer_Location', 'Rating', 'Year', 'Month', 'Month_Year', 'Rating_Valid', 'Model_Text', 'Clean_Text', 'Language', 'Is_English', 'Word_Count', 'Language_Status', 'Language_Final', 'Review_ID', 'Translated_Text', 'Translation_Status', 'Anger', 'Disgust', 'Fear', 'Joy', 'Neutral', 'Sadness', 'Surprise', 'Dominant_Emotion', 'Dominant_Emotion_Score']

Candidate columns:
['City', 'Previous_Month', 'Month_Year', 'Previous_Review_Count', 'Review_Count', 'JS_Drift', 'Cosine_Drift', 'Robust_Z', 'Drift_Category', 'Dominant_Emotion', 'Dominant_Change', 'Dominant_Direction', 'Delta_Anger', 'Delta_Disgust', 'Delta_Fear', 'Delta_Joy', 'Delta_Neutral', 'Delta_Sadness', 'Delta_Surprise']


In [22]:
# ============================================================
# STEP 3: PREPARE DATE FIELDS
# ============================================================

reviews["Month_Year"] = pd.to_datetime(
    reviews["Month_Year"],
    errors="coerce"
)

candidates["Previous_Month"] = pd.to_datetime(
    candidates["Previous_Month"],
    errors="coerce"
)

candidates["Month_Year"] = pd.to_datetime(
    candidates["Month_Year"],
    errors="coerce"
)

print("Missing review Month_Year values:",
      reviews["Month_Year"].isna().sum())

print("Missing candidate Previous_Month values:",
      candidates["Previous_Month"].isna().sum())

print("Missing candidate Month_Year values:",
      candidates["Month_Year"].isna().sum())

Missing review Month_Year values: 0
Missing candidate Previous_Month values: 0
Missing candidate Month_Year values: 0


In [23]:
# ============================================================
# STEP 4: VERIFY EVENT-MAPPING CANDIDATES
# ============================================================

print("Total candidate episodes:", len(candidates))

print("\nCandidates by drift category:")
print(candidates["Drift_Category"].value_counts())

print("\nCandidates by city:")
print(
    candidates["City"]
    .value_counts()
    .sort_index()
)

display(
    candidates[
        [
            "City",
            "Previous_Month",
            "Month_Year",
            "Previous_Review_Count",
            "Review_Count",
            "JS_Drift",
            "Robust_Z",
            "Drift_Category",
            "Dominant_Emotion",
            "Dominant_Direction"
        ]
    ]
)

Total candidate episodes: 12

Candidates by drift category:
Drift_Category
Notable Drift    10
Extreme Drift     2
Name: count, dtype: int64

Candidates by city:
City
Bali         1
Barcelona    1
Cairo        2
Dubai        1
Marrakech    4
Miami        1
Paris        1
Rome         1
Name: count, dtype: int64


,City,Previous_Month,Month_Year,Previous_Review_Count,Review_Count,JS_Drift,Robust_Z,Drift_Category,Dominant_Emotion,Dominant_Direction
0,Miami,2020-11-01,2020-12-01,10.0,13,0.423731,4.599706,Extreme Drift,Disgust,Decrease
1,Marrakech,2024-11-01,2024-12-01,23.0,24,0.297725,4.523351,Extreme Drift,Joy,Decrease
2,Paris,2021-03-01,2021-04-01,10.0,16,0.272184,3.365465,Notable Drift,Neutral,Increase
3,Barcelona,2022-07-01,2022-08-01,78.0,12,0.298757,3.195793,Notable Drift,Neutral,Increase
4,Marrakech,2021-10-01,2021-11-01,25.0,15,0.237527,3.127093,Notable Drift,Joy,Decrease
5,Cairo,2023-03-01,2023-04-01,20.0,21,0.287279,2.850124,Notable Drift,Joy,Decrease
6,Cairo,2020-08-01,2020-09-01,10.0,13,0.283813,2.782004,Notable Drift,Neutral,Decrease
7,Bali,2022-11-01,2022-12-01,20.0,68,0.303915,2.771473,Notable Drift,Joy,Increase
8,Dubai,2022-07-01,2022-08-01,29.0,13,0.243619,2.761816,Notable Drift,Surprise,Increase
9,Marrakech,2019-06-01,2019-07-01,23.0,12,0.216024,2.628348,Notable Drift,Joy,Decrease


## Step 5 — Extract Reviews for Each Candidate Drift Episode

For each candidate emotional-drift episode, reviews from the previous month and the detected drift month are extracted.

This creates the review-level evidence required for subsequent analysis of:

- emotion changes,
- rating changes,
- attraction composition,
- review text and keywords.

Each review is assigned an event identifier and a period label (`Before` or `After`) so that all 12 candidate episodes can be analyzed using the same procedure.

In [24]:
# ============================================================
# STEP 5: EXTRACT BEFORE/AFTER REVIEWS FOR ALL CANDIDATES
# ============================================================

candidate_review_sets = []

for idx, event in candidates.iterrows():

    city = event["City"]
    previous_month = event["Previous_Month"]
    current_month = event["Month_Year"]

    # Reviews from previous month
    before = reviews[
        (reviews["City"] == city) &
        (reviews["Month_Year"] == previous_month)
    ].copy()

    before["Period"] = "Before"

    # Reviews from drift month
    after = reviews[
        (reviews["City"] == city) &
        (reviews["Month_Year"] == current_month)
    ].copy()

    after["Period"] = "After"

    # Unique identifier for candidate episode
    event_id = (
        f"{city}_"
        f"{previous_month.strftime('%Y-%m')}_"
        f"{current_month.strftime('%Y-%m')}"
    )

    before["Event_ID"] = event_id
    after["Event_ID"] = event_id

    before["Drift_Category"] = event["Drift_Category"]
    after["Drift_Category"] = event["Drift_Category"]

    before["Event_Robust_Z"] = event["Robust_Z"]
    after["Event_Robust_Z"] = event["Robust_Z"]

    candidate_review_sets.append(before)
    candidate_review_sets.append(after)

# Combine all extracted reviews
candidate_reviews = pd.concat(
    candidate_review_sets,
    ignore_index=True
)

print("Total extracted review rows:", len(candidate_reviews))
print("Unique candidate episodes:",
      candidate_reviews["Event_ID"].nunique())

display(
    candidate_reviews[
        [
            "Event_ID",
            "City",
            "Month_Year",
            "Period",
            "Drift_Category",
            "Event_Robust_Z"
        ]
    ].head(10)
)

Total extracted review rows: 561
Unique candidate episodes: 12


,Event_ID,City,Month_Year,Period,Drift_Category,Event_Robust_Z
0,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
1,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
2,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
3,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
4,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
5,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
6,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
7,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
8,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706
9,Miami_2020-11_2020-12,Miami,2020-11-01,Before,Extreme Drift,4.599706


In [25]:
# ============================================================
# STEP 6: VALIDATE EXTRACTED REVIEW COUNTS
# ============================================================

extracted_counts = (
    candidate_reviews
    .groupby(
        ["Event_ID", "City", "Period"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Rename period columns
extracted_counts = extracted_counts.rename(
    columns={
        "Before": "Extracted_Before",
        "After": "Extracted_After"
    }
)

# Expected counts from candidate table
expected_counts = candidates[
    [
        "City",
        "Previous_Month",
        "Month_Year",
        "Previous_Review_Count",
        "Review_Count"
    ]
].copy()

expected_counts["Event_ID"] = (
    expected_counts["City"] + "_" +
    expected_counts["Previous_Month"].dt.strftime("%Y-%m") + "_" +
    expected_counts["Month_Year"].dt.strftime("%Y-%m")
)

validation = expected_counts.merge(
    extracted_counts[
        [
            "Event_ID",
            "Extracted_Before",
            "Extracted_After"
        ]
    ],
    on="Event_ID",
    how="left"
)

validation["Before_Match"] = (
    validation["Previous_Review_Count"]
    == validation["Extracted_Before"]
)

validation["After_Match"] = (
    validation["Review_Count"]
    == validation["Extracted_After"]
)

display(validation)

print(
    "\nAll before counts match:",
    validation["Before_Match"].all()
)

print(
    "All after counts match:",
    validation["After_Match"].all()
)

,City,Previous_Month,Month_Year,Previous_Review_Count,Review_Count,Event_ID,Extracted_Before,Extracted_After,Before_Match,After_Match
0,Miami,2020-11-01,2020-12-01,10.0,13,Miami_2020-11_2020-12,10,13,True,True
1,Marrakech,2024-11-01,2024-12-01,23.0,24,Marrakech_2024-11_2024-12,23,24,True,True
2,Paris,2021-03-01,2021-04-01,10.0,16,Paris_2021-03_2021-04,10,16,True,True
3,Barcelona,2022-07-01,2022-08-01,78.0,12,Barcelona_2022-07_2022-08,78,12,True,True
4,Marrakech,2021-10-01,2021-11-01,25.0,15,Marrakech_2021-10_2021-11,25,15,True,True
5,Cairo,2023-03-01,2023-04-01,20.0,21,Cairo_2023-03_2023-04,20,21,True,True
6,Cairo,2020-08-01,2020-09-01,10.0,13,Cairo_2020-08_2020-09,10,13,True,True
7,Bali,2022-11-01,2022-12-01,20.0,68,Bali_2022-11_2022-12,20,68,True,True
8,Dubai,2022-07-01,2022-08-01,29.0,13,Dubai_2022-07_2022-08,29,13,True,True
9,Marrakech,2019-06-01,2019-07-01,23.0,12,Marrakech_2019-06_2019-07,23,12,True,True



All before counts match: True
All after counts match: True


## Step 7 — Internal Diagnostic Summary

The emotional-drift episodes analyzed in this notebook were already identified in the previous emotional-drift analysis. Therefore, the objective here is not to detect drift again, but to investigate the review-level characteristics associated with each selected episode.

For each candidate transition, three forms of internal evidence are summarized:

- the largest change in emotion probability, obtained from the previously calculated drift results;
- the change in average review rating;
- the degree of overlap between attractions reviewed before and after the transition.

These diagnostics provide context for interpreting each episode before investigating possible external events. They also help identify cases where an observed city-level emotional shift may be influenced by changes in the composition of reviewed attractions.

In [26]:
# ============================================================
# STEP 7: CREATE INTERNAL DIAGNOSTIC SUMMARY
# ============================================================

emotion_cols = [
    "Anger",
    "Disgust",
    "Fear",
    "Joy",
    "Neutral",
    "Sadness",
    "Surprise"
]

delta_cols = [
    f"Delta_{emotion}"
    for emotion in emotion_cols
]

diagnostic_rows = []

for _, event in candidates.iterrows():

    city = event["City"]
    previous_month = event["Previous_Month"]
    current_month = event["Month_Year"]

    event_id = (
        f"{city}_"
        f"{previous_month.strftime('%Y-%m')}_"
        f"{current_month.strftime('%Y-%m')}"
    )

    event_reviews = candidate_reviews[
        candidate_reviews["Event_ID"] == event_id
    ]

    before = event_reviews[
        event_reviews["Period"] == "Before"
    ]

    after = event_reviews[
        event_reviews["Period"] == "After"
    ]

    # --------------------------------------------------------
    # 1. Largest emotion change
    # Already calculated in previous drift analysis
    # --------------------------------------------------------

    emotion_changes = {
        emotion: event[f"Delta_{emotion}"]
        for emotion in emotion_cols
    }

    largest_emotion = max(
        emotion_changes,
        key=lambda x: abs(emotion_changes[x])
    )

    largest_emotion_change = emotion_changes[
        largest_emotion
    ]

    emotion_direction = (
        "Increase"
        if largest_emotion_change > 0
        else "Decrease"
    )

    # --------------------------------------------------------
    # 2. Rating change
    # --------------------------------------------------------

    before_rating = before["Rating"].mean()
    after_rating = after["Rating"].mean()

    rating_change = (
        after_rating - before_rating
    )

    # --------------------------------------------------------
    # 3. Attraction overlap
    # --------------------------------------------------------

    before_places = set(
        before["Place"].dropna().unique()
    )

    after_places = set(
        after["Place"].dropna().unique()
    )

    common_places = (
        before_places & after_places
    )

    all_places = (
        before_places | after_places
    )

    attraction_overlap = (
        len(common_places) / len(all_places)
        if len(all_places) > 0
        else np.nan
    )

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    diagnostic_rows.append({

        "Event_ID": event_id,
        "City": city,

        "Previous_Month": previous_month,
        "Drift_Month": current_month,

        "Robust_Z": event["Robust_Z"],
        "Drift_Category": event["Drift_Category"],

        "Main_Emotion_Change": largest_emotion,
        "Emotion_Direction": emotion_direction,
        "Emotion_Change": largest_emotion_change,

        "Before_Mean_Rating": before_rating,
        "After_Mean_Rating": after_rating,
        "Rating_Change": rating_change,

        "Before_Reviews": len(before),
        "After_Reviews": len(after),

        "Before_Attractions": len(before_places),
        "After_Attractions": len(after_places),
        "Common_Attractions": len(common_places),
        "Attraction_Overlap": attraction_overlap
    })


internal_diagnostic_summary = pd.DataFrame(
    diagnostic_rows
)

internal_diagnostic_summary = (
    internal_diagnostic_summary
    .sort_values(
        "Robust_Z",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    internal_diagnostic_summary.round(3)
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Before_Mean_Rating,After_Mean_Rating,Rating_Change,Before_Reviews,After_Reviews,Before_Attractions,After_Attractions,Common_Attractions,Attraction_Overlap
0,Miami_2020-11_2020-12,Miami,2020-11-01,2020-12-01,4.600,Extreme Drift,Disgust,Decrease,-0.365,3.100,3.846,0.746,10,13,3,3,3,1.000
1,Marrakech_2024-11_2024-12,Marrakech,2024-11-01,2024-12-01,4.523,Extreme Drift,Joy,Decrease,-0.267,3.870,2.542,-1.328,23,24,3,2,2,0.667
2,Paris_2021-03_2021-04,Paris,2021-03-01,2021-04-01,3.365,Notable Drift,Neutral,Increase,0.290,4.200,3.375,-0.825,10,16,5,4,4,0.800
3,Barcelona_2022-07_2022-08,Barcelona,2022-07-01,2022-08-01,3.196,Notable Drift,Neutral,Increase,0.293,4.282,5.000,0.718,78,12,4,2,2,0.500
4,Marrakech_2021-10_2021-11,Marrakech,2021-10-01,2021-11-01,3.127,Notable Drift,Joy,Decrease,-0.201,4.120,3.867,-0.253,25,15,5,4,4,0.800
5,Cairo_2023-03_2023-04,Cairo,2023-03-01,2023-04-01,2.850,Notable Drift,Joy,Decrease,-0.258,4.300,4.000,-0.300,20,21,4,4,3,0.600
6,Cairo_2020-08_2020-09,Cairo,2020-08-01,2020-09-01,2.782,Notable Drift,Neutral,Decrease,-0.307,4.700,4.769,0.069,10,13,5,2,2,0.400
7,Bali_2022-11_2022-12,Bali,2022-11-01,2022-12-01,2.771,Notable Drift,Joy,Increase,0.335,3.900,4.412,0.512,20,68,5,5,5,1.000
8,Dubai_2022-07_2022-08,Dubai,2022-07-01,2022-08-01,2.762,Notable Drift,Surprise,Increase,0.246,4.414,4.538,0.125,29,13,4,2,2,0.500
9,Marrakech_2019-06_2019-07,Marrakech,2019-06-01,2019-07-01,2.628,Notable Drift,Joy,Decrease,-0.260,4.217,3.833,-0.384,23,12,2,1,1,0.500


## Step 8 — Review-Level Evidence

To understand the detected emotional-drift episodes, the underlying reviews are examined before searching for external events.

For each candidate episode, the review-level evidence includes:

- attraction,
- rating,
- review text,
- dominant emotion,
- dominant-emotion confidence,
- and the seven emotion probabilities.

This allows the statistical drift patterns to be interpreted using the actual tourist experiences described in the reviews.

External events are investigated only after examining this internal evidence, reducing the risk of associating detected emotional changes with unrelated real-world events.

In [27]:
# ============================================================
# STEP 8A: IDENTIFY REVIEW TEXT COLUMNS
# ============================================================

text_candidates = [
    col for col in candidate_reviews.columns
    if any(
        term in col.lower()
        for term in [
            "review",
            "text",
            "comment",
            "translated",
            "content",
            "description"
        ]
    )
]

print("Possible review-text columns:")
print(text_candidates)

Possible review-text columns:
['Review_Text', 'Review_Date', 'Reviewer_Location', 'Model_Text', 'Clean_Text', 'Review_ID', 'Translated_Text']


In [28]:
# ============================================================
# STEP 8B: EXTRACT REVIEWS MOST RELEVANT TO EACH DRIFT
# ============================================================

review_evidence_rows = []

for _, event in internal_diagnostic_summary.iterrows():

    event_id = event["Event_ID"]
    main_emotion = event["Main_Emotion_Change"]
    direction = event["Emotion_Direction"]

    event_reviews = candidate_reviews[
        candidate_reviews["Event_ID"] == event_id
    ].copy()

    # Focus primarily on the drift month
    after_reviews = event_reviews[
        event_reviews["Period"] == "After"
    ].copy()

    # Select reviews with strongest probability
    # for the emotion involved in the drift
    after_reviews = after_reviews.sort_values(
        main_emotion,
        ascending=False
    )

    # Keep top 5 reviews
    top_reviews = after_reviews.head(5)

    for _, review in top_reviews.iterrows():

        review_evidence_rows.append({

            "Event_ID": event_id,
            "City": event["City"],

            "Main_Emotion_Change": main_emotion,
            "Emotion_Direction": direction,

            "Place": review["Place"],
            "Rating": review["Rating"],

            "Dominant_Emotion":
                review["Dominant_Emotion"],

            "Target_Emotion_Probability":
                review[main_emotion],

            "Review_Text":
                review["Translated_Text"]
        })


review_evidence = pd.DataFrame(
    review_evidence_rows
)

display(review_evidence)

,Event_ID,City,Main_Emotion_Change,Emotion_Direction,Place,Rating,Dominant_Emotion,Target_Emotion_Probability,Review_Text
0,Miami_2020-11_2020-12,Miami,Disgust,Decrease,Miami Seaquarium,3,neutral,0.141157,Would not recommend
1,Miami_2020-11_2020-12,Miami,Disgust,Decrease,Miami Seaquarium,1,fear,0.026589,Do Not Go to this Aquarium!!!
2,Miami_2020-11_2020-12,Miami,Disgust,Decrease,Bayside Marketplace,5,neutral,0.024867,Lots to do.
3,Miami_2020-11_2020-12,Miami,Disgust,Decrease,Miami Seaquarium,2,neutral,0.012026,Not impressive at all but we love aquariums so...
4,Miami_2020-11_2020-12,Miami,Disgust,Decrease,Miami Beach Boardwalk,3,neutral,0.011413,Lots of Work Being Done
5,Marrakech_2024-11_2024-12,Marrakech,Joy,Decrease,Saadian Tombs,5,joy,0.983366,Very beautiful and peaceful
6,Marrakech_2024-11_2024-12,Marrakech,Joy,Decrease,Saadian Tombs,4,neutral,0.295923,Interesting tomb visit with a guide
7,Marrakech_2024-11_2024-12,Marrakech,Joy,Decrease,Jemaa el-Fnaa,5,fear,0.181326,Unforgettable experience
8,Marrakech_2024-11_2024-12,Marrakech,Joy,Decrease,Saadian Tombs,1,neutral,0.102622,"Biggest tourist trap, AVOID!"
9,Marrakech_2024-11_2024-12,Marrakech,Joy,Decrease,Jemaa el-Fnaa,3,neutral,0.083595,Timo at food stand 75


In [29]:
# ============================================================
# STEP 8C: BEFORE/AFTER REVIEW EVIDENCE
# ============================================================

comparison_rows = []

for _, event in internal_diagnostic_summary.iterrows():

    event_id = event["Event_ID"]
    main_emotion = event["Main_Emotion_Change"]

    event_reviews = candidate_reviews[
        candidate_reviews["Event_ID"] == event_id
    ]

    for period in ["Before", "After"]:

        period_reviews = event_reviews[
            event_reviews["Period"] == period
        ].copy()

        # Reviews most strongly expressing the emotion
        top_reviews = (
            period_reviews
            .sort_values(
                main_emotion,
                ascending=False
            )
            .head(3)
        )

        for _, review in top_reviews.iterrows():

            comparison_rows.append({

                "Event_ID": event_id,
                "City": event["City"],

                "Period": period,

                "Main_Emotion":
                    main_emotion,

                "Place":
                    review["Place"],

                "Rating":
                    review["Rating"],

                "Emotion_Probability":
                    review[main_emotion],

                "Review_Text":
                    review["Translated_Text"]
            })


review_text_comparison = pd.DataFrame(
    comparison_rows
)

display(review_text_comparison)

,Event_ID,City,Period,Main_Emotion,Place,Rating,Emotion_Probability,Review_Text
0,Miami_2020-11_2020-12,Miami,Before,Disgust,Bayside Marketplace,3,0.913030,Drunk people
1,Miami_2020-11_2020-12,Miami,Before,Disgust,Miami Seaquarium,1,0.865801,Worst sea quarium ever
2,Miami_2020-11_2020-12,Miami,Before,Disgust,Miami Seaquarium,1,0.852902,Humans entertained by other animals cruelty
3,Miami_2020-11_2020-12,Miami,After,Disgust,Miami Seaquarium,3,0.141157,Would not recommend
4,Miami_2020-11_2020-12,Miami,After,Disgust,Miami Seaquarium,1,0.026589,Do Not Go to this Aquarium!!!
...,...,...,...,...,...,...,...,...
67,Marrakech_2024-08_2024-09,Marrakech,Before,Neutral,Jemaa el-Fnaa,4,0.876371,Other High Place of Marrakech
68,Marrakech_2024-08_2024-09,Marrakech,Before,Neutral,Le Jardin Secret,4,0.827963,"A name, a guarantee"
69,Marrakech_2024-08_2024-09,Marrakech,After,Neutral,Saadian Tombs,4,0.597411,Interesting part of the history of Morocco.
70,Marrakech_2024-08_2024-09,Marrakech,After,Neutral,Le Jardin Secret,2,0.493800,We can dispense with it.


In [30]:
# ============================================================
# STEP 8D: DISPLAY EXTREME-DRIFT REVIEW EVIDENCE
# ============================================================

extreme_event_ids = internal_diagnostic_summary.loc[
    internal_diagnostic_summary["Drift_Category"]
    == "Extreme Drift",
    "Event_ID"
]

extreme_review_evidence = review_text_comparison[
    review_text_comparison["Event_ID"].isin(
        extreme_event_ids
    )
]

display(extreme_review_evidence)

,Event_ID,City,Period,Main_Emotion,Place,Rating,Emotion_Probability,Review_Text
0,Miami_2020-11_2020-12,Miami,Before,Disgust,Bayside Marketplace,3,0.913030,Drunk people
1,Miami_2020-11_2020-12,Miami,Before,Disgust,Miami Seaquarium,1,0.865801,Worst sea quarium ever
2,Miami_2020-11_2020-12,Miami,Before,Disgust,Miami Seaquarium,1,0.852902,Humans entertained by other animals cruelty
3,Miami_2020-11_2020-12,Miami,After,Disgust,Miami Seaquarium,3,0.141157,Would not recommend
4,Miami_2020-11_2020-12,Miami,After,Disgust,Miami Seaquarium,1,0.026589,Do Not Go to this Aquarium!!!
5,Miami_2020-11_2020-12,Miami,After,Disgust,Bayside Marketplace,5,0.024867,Lots to do.
6,Marrakech_2024-11_2024-12,Marrakech,Before,Joy,Le Jardin Secret,5,0.980377,A pleasant place for a beer on a hot afternoon
7,Marrakech_2024-11_2024-12,Marrakech,Before,Joy,Le Jardin Secret,5,0.978287,A beautiful garden in the streets of the souk.
8,Marrakech_2024-11_2024-12,Marrakech,Before,Joy,Le Jardin Secret,4,0.965543,A pleasant oasis of calm.
9,Marrakech_2024-11_2024-12,Marrakech,After,Joy,Saadian Tombs,5,0.983366,Very beautiful and peaceful


## Step 9 — Review Evidence Behind the Drift

The previous analysis identifies the statistical direction of emotional change, but external-event interpretation requires examining what tourists actually described.

For each candidate drift month, reviews with the strongest positive and negative emotional signals are therefore extracted.

Negative intensity is represented by the combined probability of anger, disgust, fear and sadness, while positive intensity is represented by joy.

These reviews are used as qualitative evidence to identify recurring experiences or themes that may help explain the detected emotional shift. They are not treated as proof of an external event.

In [31]:
# ============================================================
# STEP 9: EXTRACT REVIEWS DRIVING THE DRIFT
# ============================================================

drift_review_evidence = []

for _, event in internal_diagnostic_summary.iterrows():

    event_id = event["Event_ID"]

    after = candidate_reviews[
        (candidate_reviews["Event_ID"] == event_id) &
        (candidate_reviews["Period"] == "After")
    ].copy()

    # Combined negative-emotion probability
    after["Negative_Signal"] = (
        after["Anger"] +
        after["Disgust"] +
        after["Fear"] +
        after["Sadness"]
    )

    # Three strongest negative reviews
    negative_reviews = (
        after
        .sort_values("Negative_Signal", ascending=False)
        .head(3)
    )

    # Three strongest joyful reviews
    positive_reviews = (
        after
        .sort_values("Joy", ascending=False)
        .head(3)
    )

    for evidence_type, selected in [
        ("Strongest Negative", negative_reviews),
        ("Strongest Positive", positive_reviews)
    ]:

        for _, review in selected.iterrows():

            drift_review_evidence.append({
                "Event_ID": event_id,
                "City": event["City"],
                "Drift_Category": event["Drift_Category"],
                "Evidence_Type": evidence_type,
                "Place": review["Place"],
                "Rating": review["Rating"],
                "Dominant_Emotion": review["Dominant_Emotion"],
                "Negative_Signal": review["Negative_Signal"],
                "Joy": review["Joy"],
                "Review_Text": review["Translated_Text"]
            })


drift_review_evidence = pd.DataFrame(
    drift_review_evidence
)

display(drift_review_evidence)

,Event_ID,City,Drift_Category,Evidence_Type,Place,Rating,Dominant_Emotion,Negative_Signal,Joy,Review_Text
0,Miami_2020-11_2020-12,Miami,Extreme Drift,Strongest Negative,Miami Seaquarium,1,fear,0.970689,0.002404,Do Not Go to this Aquarium!!!
1,Miami_2020-11_2020-12,Miami,Extreme Drift,Strongest Negative,Miami Beach Boardwalk,5,sadness,0.804728,0.032544,After Christmas Vacation
2,Miami_2020-11_2020-12,Miami,Extreme Drift,Strongest Negative,Miami Seaquarium,3,neutral,0.502531,0.007823,Would not recommend
3,Miami_2020-11_2020-12,Miami,Extreme Drift,Strongest Positive,Bayside Marketplace,5,joy,0.007200,0.963559,This place was absolutely gorgeous! Pretty sce...
4,Miami_2020-11_2020-12,Miami,Extreme Drift,Strongest Positive,Bayside Marketplace,5,joy,0.019644,0.905140,Great place for a drink on the water!
...,...,...,...,...,...,...,...,...,...,...
67,Marrakech_2024-08_2024-09,Marrakech,Notable Drift,Strongest Negative,Le Jardin Secret,5,sadness,0.942219,0.002017,NOT TO BE MISSED
68,Marrakech_2024-08_2024-09,Marrakech,Notable Drift,Strongest Negative,Jemaa el-Fnaa,1,sadness,0.904682,0.003138,I would never go back to this place again (unb...
69,Marrakech_2024-08_2024-09,Marrakech,Notable Drift,Strongest Positive,Le Jardin Secret,5,joy,0.005052,0.983941,"I liked this place a lot. Genuinely beautiful,..."
70,Marrakech_2024-08_2024-09,Marrakech,Notable Drift,Strongest Positive,Le Jardin Secret,5,joy,0.010607,0.955280,Beautiful garden inside the medina


In [32]:
# ============================================================
# STEP 9B: EXTREME-DRIFT QUALITATIVE EVIDENCE
# ============================================================

extreme_ids = internal_diagnostic_summary.loc[
    internal_diagnostic_summary["Drift_Category"]
    == "Extreme Drift",
    "Event_ID"
]

display(
    drift_review_evidence[
        drift_review_evidence["Event_ID"].isin(extreme_ids)
    ][
        [
            "Event_ID",
            "Evidence_Type",
            "Place",
            "Rating",
            "Dominant_Emotion",
            "Review_Text"
        ]
    ]
)

,Event_ID,Evidence_Type,Place,Rating,Dominant_Emotion,Review_Text
0,Miami_2020-11_2020-12,Strongest Negative,Miami Seaquarium,1,fear,Do Not Go to this Aquarium!!!
1,Miami_2020-11_2020-12,Strongest Negative,Miami Beach Boardwalk,5,sadness,After Christmas Vacation
2,Miami_2020-11_2020-12,Strongest Negative,Miami Seaquarium,3,neutral,Would not recommend
3,Miami_2020-11_2020-12,Strongest Positive,Bayside Marketplace,5,joy,This place was absolutely gorgeous! Pretty sce...
4,Miami_2020-11_2020-12,Strongest Positive,Bayside Marketplace,5,joy,Great place for a drink on the water!
5,Miami_2020-11_2020-12,Strongest Positive,Miami Beach Boardwalk,4,joy,"Under construction, but nice area to walk/run"
6,Marrakech_2024-11_2024-12,Strongest Negative,Jemaa el-Fnaa,1,disgust,Animal cruelty at its best!!!
7,Marrakech_2024-11_2024-12,Strongest Negative,Jemaa el-Fnaa,4,sadness,Nice trip but I wouldn't come back
8,Marrakech_2024-11_2024-12,Strongest Negative,Jemaa el-Fnaa,2,anger,I was outraged by the abuse of the macaques. T...
9,Marrakech_2024-11_2024-12,Strongest Positive,Saadian Tombs,5,joy,Very beautiful and peaceful


## Step 10 — Review Theme and Keyword Analysis

To identify textual themes associated with each candidate drift episode, review language is compared between the month before the drift and the drift month.

TF-IDF is used as an exploratory text-analysis technique to identify terms and short phrases that are particularly representative of each period.

The objective is not to infer an external event directly from keywords. Instead, the resulting terms provide internal textual evidence that can guide the subsequent external-event investigation.

Keyword patterns are interpreted together with ratings, emotion changes, attraction composition and individual review evidence.

In [33]:
# ============================================================
# STEP 10A: TF-IDF KEYWORDS BEFORE VS AFTER
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

keyword_rows = []

for event_id in candidate_reviews["Event_ID"].unique():

    event_data = candidate_reviews[
        candidate_reviews["Event_ID"] == event_id
    ]

    city = event_data["City"].iloc[0]

    for period in ["Before", "After"]:

        period_data = event_data[
            event_data["Period"] == period
        ]

        texts = (
            period_data["Translated_Text"]
            .dropna()
            .astype(str)
            .tolist()
        )

        # Need at least two reviews for meaningful extraction
        if len(texts) < 2:
            continue

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
            max_features=100
        )

        matrix = vectorizer.fit_transform(texts)

        terms = vectorizer.get_feature_names_out()

        mean_scores = (
            matrix.mean(axis=0)
            .A1
        )

        top_indices = (
            mean_scores
            .argsort()[::-1][:10]
        )

        top_terms = [
            terms[i]
            for i in top_indices
        ]

        keyword_rows.append({
            "Event_ID": event_id,
            "City": city,
            "Period": period,
            "Top_Keywords": ", ".join(top_terms)
        })


keyword_summary = pd.DataFrame(
    keyword_rows
)

display(keyword_summary)

,Event_ID,City,Period,Top_Keywords
0,Miami_2020-11_2020-12,Miami,Before,"cruelty, nice, people, overall, overall nice, ..."
1,Miami_2020-11_2020-12,Miami,After,"lots, place, expected, recommend, aquarium, gr..."
2,Marrakech_2024-11_2024-12,Marrakech,Before,"garden, worth, time, transfer, right, expensiv..."
3,Marrakech_2024-11_2024-12,Marrakech,After,"watch, nice, place, beautiful, animal, stalls,..."
4,Paris_2021-03_2021-04,Paris,Before,"horrendous, place, place tired, love place, lo..."
5,Paris_2021-03_2021-04,Paris,After,"metro, paris, necessarily, ms, gurllll, wonder..."
6,Barcelona_2022-07_2022-08,Barcelona,Before,"beautiful, barcelona, park, worth, gaudi, visi..."
7,Barcelona_2022-07_2022-08,Barcelona,After,"barcelona, gothic, time, going time, barri got..."
8,Marrakech_2021-10_2021-11,Marrakech,Before,"nice, beautiful, marrakech, garden, stop, wort..."
9,Marrakech_2021-10_2021-11,Marrakech,After,"suggestive, palace, beautiful, unique, beautif..."


In [34]:
# ============================================================
# STEP 10B: COMPARE KEYWORDS BEFORE AND AFTER
# ============================================================

keyword_comparison = (
    keyword_summary
    .pivot(
        index=["Event_ID", "City"],
        columns="Period",
        values="Top_Keywords"
    )
    .reset_index()
)

keyword_comparison = keyword_comparison.rename(
    columns={
        "Before": "Before_Keywords",
        "After": "After_Keywords"
    }
)

keyword_comparison = keyword_comparison.merge(
    internal_diagnostic_summary[
        [
            "Event_ID",
            "Robust_Z",
            "Drift_Category",
            "Main_Emotion_Change",
            "Emotion_Direction"
        ]
    ],
    on="Event_ID",
    how="left"
)

keyword_comparison = (
    keyword_comparison
    .sort_values(
        "Robust_Z",
        ascending=False
    )
    .reset_index(drop=True)
)

display(keyword_comparison)

,Event_ID,City,After_Keywords,Before_Keywords,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction
0,Miami_2020-11_2020-12,Miami,"lots, place, expected, recommend, aquarium, gr...","cruelty, nice, people, overall, overall nice, ...",4.599706,Extreme Drift,Disgust,Decrease
1,Marrakech_2024-11_2024-12,Marrakech,"watch, nice, place, beautiful, animal, stalls,...","garden, worth, time, transfer, right, expensiv...",4.523351,Extreme Drift,Joy,Decrease
2,Paris_2021-03_2021-04,Paris,"metro, paris, necessarily, ms, gurllll, wonder...","horrendous, place, place tired, love place, lo...",3.365465,Notable Drift,Neutral,Increase
3,Barcelona_2022-07_2022-08,Barcelona,"barcelona, gothic, time, going time, barri got...","beautiful, barcelona, park, worth, gaudi, visi...",3.195793,Notable Drift,Neutral,Increase
4,Marrakech_2021-10_2021-11,Marrakech,"suggestive, palace, beautiful, unique, beautif...","nice, beautiful, marrakech, garden, stop, wort...",3.127093,Notable Drift,Joy,Decrease
5,Cairo_2023-03_2023-04,Cairo,"stunning, cairo, nice, world, guide, sensation...","mosque, lovely, cairo, stunning mosque, stunni...",2.850124,Notable Drift,Joy,Decrease
6,Cairo_2020-08_2020-09,Cairo,"place, amazing, history, disappointing, beauti...","experience, historic, unique experience, beaut...",2.782004,Notable Drift,Neutral,Decrease
7,Bali_2022-11_2022-12,Bali,"nice, monkey, experience, visit, wonderful, ri...","experience, good, worth, expecting, interestin...",2.771473,Notable Drift,Joy,Increase
8,Dubai_2022-07_2022-08,Dubai,"mall, dubai, dubai mall, world, mall world, bi...","place, good, miracle, lovely, dubai, experienc...",2.761816,Notable Drift,Surprise,Increase
9,Marrakech_2019-06_2019-07,Marrakech,"interesting, beautiful, tombs, intriguing tomb...","overpriced, beautiful, garden, medina, nice, p...",2.628348,Notable Drift,Joy,Decrease


## Step 11 — External Event Mapping

After identifying emotional-drift episodes and examining their internal review evidence, the analysis now investigates whether relevant real-world events occurred around the corresponding transition periods.

External events are considered only after the review-level analysis in order to reduce post-hoc event selection.

Potential contextual factors include:

- tourism restrictions or reopening measures,
- major cultural or sporting events,
- protests or security incidents,
- extreme weather or natural hazards,
- attraction closures or disruptions,
- transport disruptions,
- major tourism-policy changes,
- and other destination-specific events.

Temporal coincidence alone is not considered sufficient evidence.

Each possible association is evaluated using the timing of the event together with the observed emotion change, rating behavior, attraction composition, review text and keyword evidence.

The analysis identifies contextual associations rather than claiming that an external event caused the observed emotional drift.

In [35]:
# ============================================================
# STEP 11A: PREPARE EXTERNAL EVENT MAPPING TABLE
# ============================================================

external_event_mapping = (
    internal_diagnostic_summary[
        [
            "Event_ID",
            "City",
            "Previous_Month",
            "Drift_Month",
            "Robust_Z",
            "Drift_Category",
            "Main_Emotion_Change",
            "Emotion_Direction",
            "Emotion_Change",
            "Rating_Change",
            "Attraction_Overlap"
        ]
    ]
    .merge(
        keyword_comparison[
            [
                "Event_ID",
                "Before_Keywords",
                "After_Keywords"
            ]
        ],
        on="Event_ID",
        how="left"
    )
)

# Columns to be completed after external research
external_event_mapping["Possible_External_Event"] = ""
external_event_mapping["Event_Date"] = ""
external_event_mapping["Event_Category"] = ""
external_event_mapping["Review_Evidence"] = ""
external_event_mapping["External_Source"] = ""
external_event_mapping["Evidence_Strength"] = ""
external_event_mapping["Interpretation"] = ""

display(external_event_mapping)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
0,Miami_2020-11_2020-12,Miami,2020-11-01,2020-12-01,4.599706,Extreme Drift,Disgust,Decrease,-0.364703,0.746154,1.000000,"cruelty, nice, people, overall, overall nice, ...","lots, place, expected, recommend, aquarium, gr...",,,,,,,
1,Marrakech_2024-11_2024-12,Marrakech,2024-11-01,2024-12-01,4.523351,Extreme Drift,Joy,Decrease,-0.266616,-1.327899,0.666667,"garden, worth, time, transfer, right, expensiv...","watch, nice, place, beautiful, animal, stalls,...",,,,,,,
2,Paris_2021-03_2021-04,Paris,2021-03-01,2021-04-01,3.365465,Notable Drift,Neutral,Increase,0.290143,-0.825000,0.800000,"horrendous, place, place tired, love place, lo...","metro, paris, necessarily, ms, gurllll, wonder...",,,,,,,
3,Barcelona_2022-07_2022-08,Barcelona,2022-07-01,2022-08-01,3.195793,Notable Drift,Neutral,Increase,0.293166,0.717949,0.500000,"beautiful, barcelona, park, worth, gaudi, visi...","barcelona, gothic, time, going time, barri got...",,,,,,,
4,Marrakech_2021-10_2021-11,Marrakech,2021-10-01,2021-11-01,3.127093,Notable Drift,Joy,Decrease,-0.201290,-0.253333,0.800000,"nice, beautiful, marrakech, garden, stop, wort...","suggestive, palace, beautiful, unique, beautif...",,,,,,,
5,Cairo_2023-03_2023-04,Cairo,2023-03-01,2023-04-01,2.850124,Notable Drift,Joy,Decrease,-0.258351,-0.300000,0.600000,"mosque, lovely, cairo, stunning mosque, stunni...","stunning, cairo, nice, world, guide, sensation...",,,,,,,
6,Cairo_2020-08_2020-09,Cairo,2020-08-01,2020-09-01,2.782004,Notable Drift,Neutral,Decrease,-0.307213,0.069231,0.400000,"experience, historic, unique experience, beaut...","place, amazing, history, disappointing, beauti...",,,,,,,
7,Bali_2022-11_2022-12,Bali,2022-11-01,2022-12-01,2.771473,Notable Drift,Joy,Increase,0.334752,0.511765,1.000000,"experience, good, worth, expecting, interestin...","nice, monkey, experience, visit, wonderful, ri...",,,,,,,
8,Dubai_2022-07_2022-08,Dubai,2022-07-01,2022-08-01,2.761816,Notable Drift,Surprise,Increase,0.245576,0.124668,0.500000,"place, good, miracle, lovely, dubai, experienc...","mall, dubai, dubai mall, world, mall world, bi...",,,,,,,
9,Marrakech_2019-06_2019-07,Marrakech,2019-06-01,2019-07-01,2.628348,Notable Drift,Joy,Decrease,-0.260383,-0.384058,0.500000,"overpriced, beautiful, garden, medina, nice, p...","interesting, beautiful, tombs, intriguing tomb...",,,,,,,


In [36]:
# ============================================================
# STEP 11B: MAP MIAMI EXTERNAL CONTEXT
# ============================================================

event_id = "Miami_2020-11_2020-12"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Post-COVID reopening and expansion of tourism "
    "and attraction activity, including Miami Seaquarium"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "November-December 2020"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "COVID-19 tourism recovery / attraction reopening"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Disgust decreased strongly (-0.365) while mean rating "
    "increased from 3.10 to 3.85. The same three attractions "
    "were represented before and after the transition. "
    "November contained strong Seaquarium-related disgust "
    "reviews, while December contained more positive tourism "
    "experiences."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Miami Herald / CBS Miami / Miami-Dade County"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The transition coincided with the reopening and expansion "
    "of attraction and tourism activity during the COVID-19 "
    "recovery period. This provides plausible contextual "
    "alignment with the observed improvement in review emotion "
    "and ratings, but does not establish that reopening caused "
    "the emotional drift."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"]
        == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
0,Miami_2020-11_2020-12,Miami,2020-11-01,2020-12-01,4.599706,Extreme Drift,Disgust,Decrease,-0.364703,0.746154,1.0,"cruelty, nice, people, overall, overall nice, ...","lots, place, expected, recommend, aquarium, gr...",Post-COVID reopening and expansion of tourism ...,November-December 2020,COVID-19 tourism recovery / attraction reopening,Disgust decreased strongly (-0.365) while mean...,Miami Herald / CBS Miami / Miami-Dade County,Moderate,The transition coincided with the reopening an...


### Interpretation of the Miami November–December 2020 Episode

The Miami transition from November to December 2020 was identified as an **Extreme Drift** episode with a Robust Z-score of **4.60**. The largest emotional change was a substantial **decrease in Disgust (-0.365)**.

The external-event interpretation was developed only after examining the internal review evidence. Several observations were considered:

1. **Emotion change:** Disgust decreased substantially between November and December 2020.

2. **Rating behavior:** The average rating increased from **3.10 to 3.85 (+0.746)**, providing an additional indication that the December review sample reflected a more positive visitor experience overall.

3. **Attraction composition:** The same three attractions were represented in both months, producing an attraction overlap of **1.00**. Therefore, the observed emotional change cannot be explained simply by an entirely different set of attractions entering the sample.

4. **Review-level evidence:** November contained several strongly negative reviews, particularly concerning Miami Seaquarium. Examples included criticism related to animal cruelty and highly negative assessments of the attraction. In December, these strong Disgust signals were considerably weaker, while more positive experiences appeared in reviews of Bayside Marketplace and Miami Beach Boardwalk.

5. **Textual evidence:** The TF-IDF comparison also indicated a change in the language used between the two months. November contained terms associated with the strongly negative experiences, including *cruelty*, whereas the December review vocabulary contained more general tourism and recommendation-related terms.

6. **External context:** Independent historical sources indicate that Miami Seaquarium reopened on **6 November 2020** after an extended COVID-19 closure, initially operating with health restrictions and reduced activities. Its operating schedule subsequently expanded during December, while Miami also experienced increased tourism activity around the Christmas holiday period.

Taken together, the statistical, rating, attraction, textual and external evidence provides a **plausible contextual association** between the detected emotional transition and the changing tourism environment during the COVID-19 reopening period.

However, the available evidence does **not demonstrate causality**. The number of reviews was relatively small (**10 reviews in November and 13 in December**), and no evidence directly establishes that reopening conditions caused tourists to express less Disgust. The external-event association is therefore classified as **Moderate**, rather than Strong.

This case is interpreted as an example of how an extreme emotional-drift signal can coincide with a meaningful change in destination context while still requiring cautious interpretation.

In [37]:
# ============================================================
# STEP 11C: MAP MARRAKECH 2024 EXTERNAL CONTEXT
# ============================================================

event_id = "Marrakech_2024-11_2024-12"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Ongoing animal-welfare concerns involving Barbary "
    "macaques and other animals used for tourist entertainment "
    "at Jemaa el-Fnaa"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "Ongoing issue; observed in December 2024 reviews"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "Destination-specific animal-welfare issue"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Joy decreased by 0.267 and mean rating fell from "
    "3.87 to 2.54. December negative reviews were concentrated "
    "around Jemaa el-Fnaa and explicitly referred to animal "
    "cruelty and abuse of macaques. TF-IDF analysis also "
    "surfaced animal-related vocabulary in December."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "National Geographic / Born Free"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The emotional deterioration coincides with review-level "
    "concerns about animal treatment at Jemaa el-Fnaa. "
    "Independent sources confirm that exploitation of Barbary "
    "macaques for tourist entertainment is a documented and "
    "persistent issue at the square. However, no evidence was "
    "identified of a discrete new animal-welfare event beginning "
    "in December 2024. The result is therefore interpreted as "
    "contextual alignment with an ongoing destination-specific "
    "issue rather than a causal response to a newly occurring event."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"]
        == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
1,Marrakech_2024-11_2024-12,Marrakech,2024-11-01,2024-12-01,4.523351,Extreme Drift,Joy,Decrease,-0.266616,-1.327899,0.666667,"garden, worth, time, transfer, right, expensiv...","watch, nice, place, beautiful, animal, stalls,...",Ongoing animal-welfare concerns involving Barb...,Ongoing issue; observed in December 2024 reviews,Destination-specific animal-welfare issue,Joy decreased by 0.267 and mean rating fell fr...,National Geographic / Born Free,Moderate,The emotional deterioration coincides with rev...


### Interpretation of the Marrakech November–December 2024 Episode

The Marrakech transition from November to December 2024 was identified as the second **Extreme Drift** episode, with a Robust Z-score of **4.52**. The principal emotional change was a **decrease in Joy (-0.267)**.

The internal evidence provides several indications that the deterioration was meaningful. Average ratings declined substantially from **3.87 to 2.54**, while review-text analysis identified animal-related vocabulary in the December reviews. Qualitative inspection further showed that several of the strongest negative December reviews concerned **Jemaa el-Fnaa**, specifically describing perceived cruelty and mistreatment of macaques.

Unlike the Miami case, attraction composition also changed to some extent, with an attraction overlap of **0.667**. Therefore, part of the city-level emotional change may reflect differences in the attractions represented in the two monthly samples.

External research was conducted only after these patterns had been identified. Independent sources document longstanding concerns regarding Barbary macaques and other animals used for tourist entertainment at Jemaa el-Fnaa. These sources support the relevance of the animal-welfare theme identified independently within the review data.

However, the external research did **not identify a discrete new animal-welfare event beginning in December 2024**. The issue appears to have existed before and after the detected transition.

Consequently, the detected emotional drift is not interpreted as being caused by a specific December event. Instead, it is considered to show **contextual alignment with an ongoing destination-specific animal-welfare issue** that became particularly visible within the December review sample.

The association is classified as **Moderate** because the review-level and independent contextual evidence strongly agree on the underlying theme, while the evidence for a temporally distinct external event is limited.

In [38]:
# ============================================================
# STEP 11D: MAP PARIS 2021 EXTERNAL CONTEXT
# ============================================================

event_id = "Paris_2021-03_2021-04"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "COVID-19 lockdown and severe restrictions on tourism, "
    "cultural venues and public activities in Paris"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "March-April 2021"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "COVID-19 restrictions / tourism disruption"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Neutral emotion increased by 0.290 while Joy declined, "
    "and the average rating decreased from 4.20 to 3.38. "
    "Most attractions remained represented across both months "
    "(attraction overlap 0.80)."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Ville de Paris / Paris Musées"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The emotional transition coincided with intensified "
    "COVID-19 restrictions and severely constrained tourism "
    "and cultural activity in Paris. The timing and decline "
    "in ratings provide plausible contextual alignment. "
    "However, the small review sample and limited explicit "
    "COVID-related review text prevent a causal interpretation."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
2,Paris_2021-03_2021-04,Paris,2021-03-01,2021-04-01,3.365465,Notable Drift,Neutral,Increase,0.290143,-0.825,0.8,"horrendous, place, place tired, love place, lo...","metro, paris, necessarily, ms, gurllll, wonder...",COVID-19 lockdown and severe restrictions on t...,March-April 2021,COVID-19 restrictions / tourism disruption,Neutral emotion increased by 0.290 while Joy d...,Ville de Paris / Paris Musées,Moderate,The emotional transition coincided with intens...


### Interpretation of the Paris March–April 2021 Episode

The Paris transition from March to April 2021 was identified as a **Notable Drift** episode with a Robust Z-score of **3.37**. The largest emotional change was an **increase in Neutral emotion (+0.290)**, accompanied by a substantial reduction in Joy.

The average review rating also decreased from **4.20 to 3.38 (-0.825)**. This provides supporting evidence that the change in the monthly emotion profile occurred alongside a deterioration in the overall evaluation of visitor experiences.

Attraction overlap between the two months was relatively high (**0.80**), indicating that the transition cannot be attributed entirely to a completely different set of attractions being reviewed.

External research was conducted only after these internal patterns were identified. During the transition period, Paris was affected by major COVID-19 restrictions. Lockdown conditions were in force during April, while museums, cultural venues and several major tourist attractions remained closed. Major cultural sites did not begin reopening until May 2021.

The timing therefore provides a plausible contextual relationship between the detected emotional transition and the exceptional tourism environment created by the pandemic restrictions.

However, the review sample was relatively small (**10 reviews in March and 16 in April**), and the review text does not provide sufficient direct evidence that individual reviewers were specifically reacting to lockdown restrictions.

The association is consequently classified as **Moderate**. The external event has strong temporal relevance and is consistent with the direction of the observed rating and emotional changes, but the available evidence does not establish causality.

In [39]:
# ============================================================
# STEP 11E: MAP BARCELONA 2022 EXTERNAL CONTEXT
# ============================================================

event_id = "Barcelona_2022-07_2022-08"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = "No specific external event identified"

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "July-August 2022"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "No identifiable external association"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Neutral increased by 0.293 while Joy decreased by 0.225. "
    "However, review volume fell sharply from 78 to 12 reviews, "
    "the attraction overlap was only 0.50, and the mean rating "
    "increased from 4.28 to 5.00."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Barcelona Tourism Observatory / Barcelona Tourism"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "No identifiable external association"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "No sufficiently supported external event was identified "
    "that explains the July-August emotional transition. "
    "Barcelona maintained substantial tourism activity during "
    "August 2022. The sharp reduction in review volume and "
    "change in attraction composition provide a stronger "
    "alternative explanation for the observed city-level "
    "emotion-profile change. The episode is therefore retained "
    "as a statistically notable drift but is not mapped to a "
    "specific external event."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
3,Barcelona_2022-07_2022-08,Barcelona,2022-07-01,2022-08-01,3.195793,Notable Drift,Neutral,Increase,0.293166,0.717949,0.5,"beautiful, barcelona, park, worth, gaudi, visi...","barcelona, gothic, time, going time, barri got...",No specific external event identified,July-August 2022,No identifiable external association,Neutral increased by 0.293 while Joy decreased...,Barcelona Tourism Observatory / Barcelona Tourism,No identifiable external association,No sufficiently supported external event was i...


### Interpretation of the Barcelona July–August 2022 Episode

The Barcelona transition from July to August 2022 was identified as a **Notable Drift** episode with a Robust Z-score of **3.20**. The principal emotional change was an **increase in Neutral emotion (+0.293)**, accompanied by a decrease in Joy.

However, several characteristics distinguish this episode from the previously examined cases.

First, review volume decreased substantially from **78 reviews in July to only 12 in August**. Consequently, the August monthly emotion profile was estimated from a considerably smaller sample.

Second, attraction overlap between the two months was only **0.50**, indicating a meaningful change in the attractions represented in the review sample. The textual evidence also reflects this compositional difference, with July containing terms associated with parks and Gaudí attractions while August contained more references to the Gothic area.

Third, the average rating increased from **4.28 to 5.00** despite the decrease in Joy and increase in Neutral emotion. Therefore, the emotional drift does not correspond to a straightforward deterioration in visitor satisfaction.

External research was conducted after these internal characteristics were identified. Tourism indicators show that Barcelona continued to experience substantial tourism activity during August 2022, with most cultural facilities operating and high levels of transport, hotel and cruise activity.

No sufficiently specific external event was identified that could be defensibly associated with the observed emotional transition.

The episode is therefore classified as having **No Identifiable External Association**. The sharp reduction in review volume and change in attraction composition provide a more plausible explanation for the observed monthly emotion-profile variation than a specific external event.

This result demonstrates that a statistically notable emotional drift should not automatically be interpreted as evidence of a real-world event. Internal sampling and attraction-composition effects must also be considered.

In [40]:
# ============================================================
# STEP 11F: MAP MARRAKECH 2021 EXTERNAL CONTEXT
# ============================================================

event_id = "Marrakech_2021-10_2021-11"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Progressive COVID-19 international flight restrictions "
    "affecting Morocco during October-November 2021"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "20 October-29 November 2021"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "COVID-19 travel restrictions / tourism disruption"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Joy decreased by 0.201 while Neutral increased by 0.150 "
    "and mean rating declined from 4.12 to 3.87. Review volume "
    "also decreased from 25 to 15, while attraction overlap "
    "remained relatively high at 0.80."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Moroccan Ministry of Tourism / Euronews"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The emotional transition occurred during a period of "
    "progressively tightening international travel restrictions. "
    "Morocco suspended flights with several major European "
    "markets from October 2021, followed by restrictions involving "
    "France and ultimately all incoming passenger flights at the "
    "end of November. This provides plausible tourism-related "
    "context for the detected drift, but the review evidence does "
    "not directly establish that these restrictions produced the "
    "observed emotional change."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
4,Marrakech_2021-10_2021-11,Marrakech,2021-10-01,2021-11-01,3.127093,Notable Drift,Joy,Decrease,-0.20129,-0.253333,0.8,"nice, beautiful, marrakech, garden, stop, wort...","suggestive, palace, beautiful, unique, beautif...",Progressive COVID-19 international flight rest...,20 October-29 November 2021,COVID-19 travel restrictions / tourism disruption,Joy decreased by 0.201 while Neutral increased...,Moroccan Ministry of Tourism / Euronews,Moderate,The emotional transition occurred during a per...


### Interpretation of the Marrakech October–November 2021 Episode

The Marrakech transition from October to November 2021 was identified as a **Notable Drift** episode with a Robust Z-score of **3.13**. The principal emotional change was a **decrease in Joy (-0.201)**, accompanied by increases in Neutral and Surprise.

The average rating also declined moderately from **4.12 to 3.87**, while review volume decreased from **25 to 15 reviews**. Attraction overlap remained relatively high (**0.80**), suggesting that the transition was not solely the result of a completely different set of attractions being reviewed.

External research identified a period of progressively increasing international travel restrictions affecting Morocco. On 20 October 2021, direct flights with several important European markets, including the United Kingdom, Germany and the Netherlands, were suspended because of the COVID-19 situation. Restrictions intensified during late November, including the suspension of flights with France and, from 29 November, the temporary suspension of all incoming direct passenger flights in response to the emerging Omicron variant.

These measures provide a relevant external tourism context for the October–November transition, particularly for an internationally dependent tourism destination such as Marrakech.

Nevertheless, the timing requires cautious interpretation. Some of the most restrictive measures were introduced only at the end of November, and the review text does not provide sufficient direct evidence that individual reviewers were responding specifically to flight restrictions.

The episode is therefore classified as a **Moderate contextual association**. The external disruption is temporally relevant and consistent with a period of tourism uncertainty, but it cannot be considered a demonstrated cause of the observed emotional drift.

In [41]:
# ============================================================
# STEP 11G: MAP CAIRO MARCH-APRIL 2023 CONTEXT
# ============================================================

event_id = "Cairo_2023-03_2023-04"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Ramadan 2023 and associated changes in daily operating "
    "patterns, transport and attraction schedules"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "23 March-20 April 2023"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "Religious period / tourism operating environment"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Joy decreased by 0.258 while Disgust, Surprise and Anger "
    "increased. Mean rating declined slightly from 4.30 to 4.00. "
    "Review volume remained stable at 20 to 21 reviews, while "
    "attraction overlap was 0.60."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Egypt State Information Service / "
    "Ahram Online / Ministry of Tourism and Antiquities"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Weak"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The March-April transition coincided closely with Ramadan, "
    "which altered operating schedules and daily activity patterns "
    "in Cairo. However, the review text does not directly connect "
    "the emotional change to Ramadan, the rating decline was small, "
    "and Egypt recorded very strong tourism activity in April. "
    "Ramadan is therefore retained only as a possible contextual "
    "factor rather than an explanation of the detected drift."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
5,Cairo_2023-03_2023-04,Cairo,2023-03-01,2023-04-01,2.850124,Notable Drift,Joy,Decrease,-0.258351,-0.3,0.6,"mosque, lovely, cairo, stunning mosque, stunni...","stunning, cairo, nice, world, guide, sensation...",Ramadan 2023 and associated changes in daily o...,23 March-20 April 2023,Religious period / tourism operating environment,"Joy decreased by 0.258 while Disgust, Surprise...",Egypt State Information Service / Ahram Online...,Weak,The March-April transition coincided closely w...


### Interpretation of the Cairo March–April 2023 Episode

The Cairo transition from March to April 2023 was identified as a **Notable Drift** episode with a Robust Z-score of **2.85**. The largest emotional change was a **decrease in Joy (-0.258)**, accompanied by increases in Disgust, Surprise and Anger.

Unlike several other candidate episodes, review volume remained relatively stable, increasing only from **20 to 21 reviews**. Average rating decreased moderately from **4.30 to 4.00**, while attraction overlap was **0.60**, indicating some change in the attractions represented between the two months.

External research identified Ramadan as the principal city-wide contextual change overlapping the transition. Ramadan began in Egypt on **23 March 2023** and continued through most of April. During this period, operating schedules across Cairo changed, including commercial opening hours, public transport schedules and visiting hours at major tourist sites.

The temporal alignment is therefore strong. However, temporal coincidence alone is insufficient to establish an association with the emotional drift. The review keywords do not provide clear evidence that tourists were reacting specifically to Ramadan-related conditions. Furthermore, Egypt recorded exceptionally high international tourist arrivals in April 2023, indicating that the period was not characterized by a general collapse in tourism activity.

For these reasons, Ramadan is retained as a **possible contextual factor**, but the association is classified as **Weak**. The evidence supports temporal overlap with a substantially different urban and tourism operating environment, but does not demonstrate that Ramadan produced the observed reduction in Joy.

This case illustrates the distinction between identifying a relevant external context and demonstrating that the context explains an emotional-drift episode.

In [42]:
# ============================================================
# STEP 11H: MAP CAIRO AUGUST-SEPTEMBER 2020 CONTEXT
# ============================================================

event_id = "Cairo_2020-08_2020-09"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Resumption of cultural tourism and nationwide reopening "
    "of museums and archaeological sites in Egypt"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "1 September 2020"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "COVID-19 reopening / cultural tourism recovery"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Neutral decreased by 0.307 while Surprise increased by "
    "0.272. Mean rating remained very positive, increasing "
    "slightly from 4.70 to 4.77. Review volume increased from "
    "10 to 13, but attraction overlap was low at 0.40."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Egypt State Information Service / "
    "Ministry of Tourism and Antiquities"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The detected emotional transition coincided exactly with "
    "Egypt's resumption of cultural tourism on 1 September 2020, "
    "when museums and archaeological sites reopened nationwide. "
    "The increase in Surprise and continued high ratings are "
    "compatible with a changing tourism environment following "
    "reopening. However, the small number of reviews and low "
    "attraction overlap indicate that sample and attraction "
    "composition may also have contributed substantially. "
    "The relationship is therefore treated as contextual "
    "alignment rather than evidence of causality."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
6,Cairo_2020-08_2020-09,Cairo,2020-08-01,2020-09-01,2.782004,Notable Drift,Neutral,Decrease,-0.307213,0.069231,0.4,"experience, historic, unique experience, beaut...","place, amazing, history, disappointing, beauti...",Resumption of cultural tourism and nationwide ...,1 September 2020,COVID-19 reopening / cultural tourism recovery,Neutral decreased by 0.307 while Surprise incr...,Egypt State Information Service / Ministry of ...,Moderate,The detected emotional transition coincided ex...


### Interpretation of the Cairo August–September 2020 Episode

The Cairo transition from August to September 2020 was identified as a **Notable Drift** episode with a Robust Z-score of **2.78**. The most substantial emotional change was a **decrease in Neutral emotion (-0.307)**, accompanied by a large **increase in Surprise (+0.272)**.

Average ratings remained highly positive, changing only slightly from **4.70 to 4.77**. Therefore, the emotional drift should not be interpreted as a deterioration in tourist satisfaction.

External research was conducted after identifying this pattern. A particularly relevant tourism-policy transition occurred exactly at the beginning of the detected drift month. On **1 September 2020**, Egypt resumed cultural tourism and reopened museums and archaeological sites nationwide following COVID-19 closures. The reopening was accompanied by health and capacity restrictions, including social distancing, face-mask requirements and visitor limits.

The exact temporal correspondence between the September reopening and the detected August–September emotional transition makes the reopening a plausible contextual factor. The increase in Surprise, together with persistently high ratings, is compatible with a changing visitor environment rather than a negative tourism shock.

However, important internal limitations remain. Only **10 reviews were available in August and 13 in September**, placing both monthly samples close to the minimum inclusion threshold. Furthermore, attraction overlap was only **0.40**, indicating that substantially different attractions contributed to the two monthly profiles.

Consequently, the episode is classified as a **Moderate contextual association**. The external event is precisely dated and directly relevant to tourism, but sample size and attraction-composition changes prevent attribution of the emotional drift to the reopening itself.

This case demonstrates why external-event mapping is performed only after examining internal evidence: even when an external event aligns almost perfectly in time, alternative explanations within the review data must still be considered.

In [43]:
# ============================================================
# STEP 11I: MAP BALI NOVEMBER-DECEMBER 2022 CONTEXT
# ============================================================

event_id = "Bali_2022-11_2022-12"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = (
    "Strong year-end increase in Bali tourism activity "
    "and international visitor arrivals"
)

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "December 2022"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "Tourism recovery / year-end tourism surge"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Joy increased strongly by 0.335 while Neutral and Sadness "
    "declined. Mean rating increased from 3.90 to 4.41 and review "
    "volume increased from 20 to 68. Attraction overlap was 1.00, "
    "indicating that the same set of attractions remained "
    "represented across both months."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = "BPS Statistics of Bali Province"

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "Moderate"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "The positive emotional transition coincided with a strong "
    "increase in Bali tourism activity. Official statistics show "
    "that direct foreign tourist arrivals increased by 31.27% "
    "from November to December 2022, accompanied by higher hotel "
    "occupancy and transport activity. The review dataset shows "
    "parallel increases in review volume, average rating and Joy. "
    "Because attraction overlap was complete, the transition is "
    "less easily explained by a completely different attraction "
    "set. Nevertheless, these parallel changes establish "
    "contextual alignment rather than causality."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
7,Bali_2022-11_2022-12,Bali,2022-11-01,2022-12-01,2.771473,Notable Drift,Joy,Increase,0.334752,0.511765,1.0,"experience, good, worth, expecting, interestin...","nice, monkey, experience, visit, wonderful, ri...",Strong year-end increase in Bali tourism activ...,December 2022,Tourism recovery / year-end tourism surge,Joy increased strongly by 0.335 while Neutral ...,BPS Statistics of Bali Province,Moderate,The positive emotional transition coincided wi...


### Interpretation of the Bali November–December 2022 Episode

The Bali transition from November to December 2022 was identified as a **Notable Drift** episode with a Robust Z-score of **2.77**. Unlike several negative-direction episodes, this transition represented a substantial movement toward a more positive emotional profile.

The largest change was an **increase in Joy (+0.335)**. Neutral and Sadness simultaneously decreased. Average rating also increased from **3.90 to 4.41**, while review volume rose substantially from **20 to 68 reviews**.

An important characteristic of this episode is that attraction overlap was **1.00**. Therefore, the November and December samples contained the same set of represented attractions. Although their relative review shares may still have differed, the emotional transition cannot be explained simply by an entirely different set of attractions appearing in December.

External research was conducted after identifying these internal patterns. Official statistics from BPS Statistics of Bali Province show that direct international tourist arrivals increased from **287,398 in November to 377,276 in December 2022**, representing a **31.27% month-to-month increase**. Hotel occupancy and several transport indicators also increased during December.

The external tourism statistics therefore move in the same direction as several independent indicators in the review dataset: tourism arrivals increased, review volume increased, average ratings improved and Joy increased substantially.

This provides a coherent contextual association between the detected emotional drift and Bali's strong year-end tourism activity and continuing tourism recovery.

However, the statistical correspondence does not demonstrate that increased tourist arrivals caused the change in review emotions. Other seasonal, visitor-composition and experience-related factors may also have contributed.

The episode is therefore classified as a **Moderate contextual association** rather than a causal relationship.

In [44]:
# ============================================================
# STEP 11J: MAP DUBAI JULY-AUGUST 2022 CONTEXT
# ============================================================

event_id = "Dubai_2022-07_2022-08"

mask = (
    external_event_mapping["Event_ID"]
    == event_id
)

external_event_mapping.loc[
    mask,
    "Possible_External_Event"
] = "No specific external event identified"

external_event_mapping.loc[
    mask,
    "Event_Date"
] = "July-August 2022"

external_event_mapping.loc[
    mask,
    "Event_Category"
] = "No identifiable external association"

external_event_mapping.loc[
    mask,
    "Review_Evidence"
] = (
    "Surprise increased by 0.246 while Joy decreased by 0.170. "
    "Mean rating remained high and increased slightly from "
    "4.41 to 4.54. Review volume fell from 29 to 13 and "
    "attraction overlap was only 0.50. August keywords also "
    "showed stronger concentration around Dubai Mall."
)

external_event_mapping.loc[
    mask,
    "External_Source"
] = (
    "Dubai Government Media Office / "
    "Dubai Department of Economy and Tourism"
)

external_event_mapping.loc[
    mask,
    "Evidence_Strength"
] = "No identifiable external association"

external_event_mapping.loc[
    mask,
    "Interpretation"
] = (
    "Dubai experienced strong summer tourism activity during "
    "both July and August 2022, including Dubai Summer Surprises "
    "and the #DubaiDestinations campaign. However, these activities "
    "span both months and no sufficiently specific external event "
    "was identified at the July-August transition. The substantial "
    "decline in review volume, low attraction overlap and stronger "
    "August concentration around Dubai Mall provide plausible "
    "sample and attraction-composition explanations. The episode "
    "is therefore retained as a statistically notable drift without "
    "attribution to a specific external event."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
8,Dubai_2022-07_2022-08,Dubai,2022-07-01,2022-08-01,2.761816,Notable Drift,Surprise,Increase,0.245576,0.124668,0.5,"place, good, miracle, lovely, dubai, experienc...","mall, dubai, dubai mall, world, mall world, bi...",No specific external event identified,July-August 2022,No identifiable external association,Surprise increased by 0.246 while Joy decrease...,Dubai Government Media Office / Dubai Departme...,No identifiable external association,Dubai experienced strong summer tourism activi...


### Interpretation of the Dubai July–August 2022 Episode

The Dubai transition from July to August 2022 was identified as a **Notable Drift** episode with a Robust Z-score of **2.76**. The principal emotional change was a substantial **increase in Surprise (+0.246)**, accompanied by a decrease in Joy.

Average ratings remained highly positive and increased slightly from **4.41 to 4.54**. Consequently, the emotional transition does not represent a straightforward deterioration in visitor satisfaction.

Several internal characteristics suggest that sample composition may have influenced the detected drift. Review volume decreased from **29 reviews in July to only 13 in August**, while attraction overlap was only **0.50**. Furthermore, the August TF-IDF results became more strongly concentrated around terms such as *mall* and *Dubai Mall*, indicating a possible shift in the types of attractions represented.

External research was conducted after these patterns were identified. Dubai was experiencing substantial summer tourism activity during this period. Dubai Summer Surprises operated across July and August, while the #DubaiDestinations summer campaign promoted indoor attractions, hotels, malls, waterparks and other summer experiences throughout both months.

These activities provide useful background context, but they do not constitute a sufficiently distinct external event occurring specifically at the July–August transition. Assigning the emotional drift to these campaigns would therefore rely mainly on temporal coincidence.

The episode is consequently classified as having **No Identifiable External Association**. The reduction in review volume and change in attraction composition provide stronger alternative explanations for the observed emotional-profile shift.

This case demonstrates that the presence of a major tourism campaign during a drift period is not sufficient for event attribution when the event spans both comparison periods and the internal review composition changes substantially.

In [45]:
# ============================================================
# STEP 11K: MAP MARRAKECH JUNE-JULY 2019 CONTEXT
# ============================================================

event_id = "Marrakech_2019-06_2019-07"

mask = external_event_mapping["Event_ID"] == event_id

external_event_mapping.loc[
    mask, "Possible_External_Event"
] = (
    "Transition to substantially hotter summer sightseeing "
    "conditions in Marrakech"
)

external_event_mapping.loc[
    mask, "Event_Date"
] = "July 2019"

external_event_mapping.loc[
    mask, "Event_Category"
] = "Seasonal weather / tourism environment"

external_event_mapping.loc[
    mask, "Review_Evidence"
] = (
    "Joy decreased by 0.260 and average rating declined from "
    "4.22 to 3.83. July reviews continued to concern outdoor "
    "tourism attractions, including tombs and other historic "
    "sites. However, review volume fell from 23 to 12 and "
    "attraction overlap was only 0.50."
)

external_event_mapping.loc[
    mask, "External_Source"
] = (
    "Historical Marrakech weather records / "
    "Marrakech Menara meteorological data"
)

external_event_mapping.loc[
    mask, "Evidence_Strength"
] = "Weak"

external_event_mapping.loc[
    mask, "Interpretation"
] = (
    "The detected decline in Joy coincided with the transition "
    "to hotter summer conditions in Marrakech. Historical weather "
    "records show higher average and maximum temperatures in July "
    "than in June, with several very hot days during July. Such "
    "conditions provide a plausible contextual factor for outdoor "
    "sightseeing experiences. However, the review text does not "
    "directly attribute dissatisfaction to heat, and the reduced "
    "review volume and low attraction overlap provide alternative "
    "explanations. The association is therefore classified as Weak."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
9,Marrakech_2019-06_2019-07,Marrakech,2019-06-01,2019-07-01,2.628348,Notable Drift,Joy,Decrease,-0.260383,-0.384058,0.5,"overpriced, beautiful, garden, medina, nice, p...","interesting, beautiful, tombs, intriguing tomb...",Transition to substantially hotter summer sigh...,July 2019,Seasonal weather / tourism environment,Joy decreased by 0.260 and average rating decl...,Historical Marrakech weather records / Marrake...,Weak,The detected decline in Joy coincided with the...


### Interpretation of the Marrakech June–July 2019 Episode

The Marrakech transition from June to July 2019 was identified as a **Notable Drift** with a Robust Z-score of **2.63**. The principal emotional change was a **decrease in Joy (-0.260)**, while the average rating declined from **4.22 to 3.83**.

After detecting this change, external environmental conditions were investigated. Historical meteorological records indicate a transition toward hotter summer conditions between June and July 2019. Average temperature increased from approximately **24.9°C to 27.0°C**, while average maximum temperature increased from approximately **29.9°C to 33.9°C**. Several particularly hot days were also recorded during July.

This environmental change is potentially relevant because Marrakech tourism frequently involves outdoor walking and visits to historic sites, gardens and the Medina. The review keywords for the analysed period also contain references to outdoor and cultural attractions.

Nevertheless, the internal evidence does not establish that temperature was responsible for the emotional change. The review texts do not provide a sufficiently explicit heat-related signal. Furthermore, review volume decreased from **23 to 12**, and attraction overlap was only **0.50**, meaning that sample and attraction composition could also have influenced the monthly emotional profiles.

The relationship is therefore classified as a **Weak contextual association**. Hotter summer conditions provide a plausible environmental context for the observed decline in Joy, but the available evidence is insufficient to attribute the emotional drift directly to weather conditions.

In [46]:
# ============================================================
# STEP 11L: MAP ROME JUNE-JULY 2024 CONTEXT
# ============================================================

event_id = "Rome_2024-06_2024-07"

mask = external_event_mapping["Event_ID"] == event_id

external_event_mapping.loc[
    mask, "Possible_External_Event"
] = (
    "Exceptional summer heatwave affecting Rome and "
    "tourist sightseeing conditions"
)

external_event_mapping.loc[
    mask, "Event_Date"
] = "July 2024"

external_event_mapping.loc[
    mask, "Event_Category"
] = "Extreme weather / tourism environment"

external_event_mapping.loc[
    mask, "Review_Evidence"
] = (
    "Joy increased by 0.243 while Neutral decreased by 0.236. "
    "Average rating declined slightly from 4.60 to 4.38. "
    "Review volume remained relatively stable at 20 versus 24. "
    "However, attraction overlap was 0.00, indicating completely "
    "different represented attraction sets between the two months."
)

external_event_mapping.loc[
    mask, "External_Source"
] = "ANSA / Reuters"

external_event_mapping.loc[
    mask, "Evidence_Strength"
] = "Weak"

external_event_mapping.loc[
    mask, "Interpretation"
] = (
    "The July emotional transition occurred during an exceptional "
    "heatwave in Rome. Contemporary reporting documented extreme "
    "temperatures, heat alerts and difficult sightseeing conditions "
    "for tourists. This provides a temporally relevant tourism "
    "environment context. However, Joy increased rather than "
    "decreased, and attraction overlap between June and July was "
    "0.00. The change in attraction composition therefore represents "
    "a major alternative explanation for the emotional drift. "
    "The external association is consequently classified as Weak "
    "and should not be interpreted causally."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
10,Rome_2024-06_2024-07,Rome,2024-06-01,2024-07-01,2.625871,Notable Drift,Joy,Increase,0.243289,-0.225,0.0,"roman, roman forum, forum, thumbs, interesting...","rome, beautiful, impressive, nice, chaos, ente...",Exceptional summer heatwave affecting Rome and...,July 2024,Extreme weather / tourism environment,Joy increased by 0.243 while Neutral decreased...,ANSA / Reuters,Weak,The July emotional transition occurred during ...


### Interpretation of the Rome June–July 2024 Episode

The Rome transition from June to July 2024 was identified as a **Notable Drift** episode with a Robust Z-score of **2.63**. The principal emotional change was an **increase in Joy (+0.243)** accompanied by a substantial decrease in Neutral emotion (-0.236). Average rating decreased slightly from **4.60 to 4.38**, while review volume remained relatively stable at 20 and 24 reviews.

External research conducted after detecting the emotional transition identified an exceptional heat episode affecting Rome during July 2024. Contemporary reports described July as the hottest July recorded in Rome, with an average maximum temperature of approximately **34.8°C**, numerous tropical nights and repeated temperature peaks approaching 37–40°C. Rome was also placed under high-level heat alerts, while contemporary reporting documented tourists attempting to cope with extreme temperatures during sightseeing.

The heatwave therefore represents a genuine and temporally relevant change in the tourism environment during the detected drift period.

However, the direction of the emotional change does not provide straightforward support for a negative heat effect. Joy increased substantially despite the extreme conditions, while average rating decreased only slightly.

More importantly, **attraction overlap between June and July was 0.00**. This means that the represented attraction sets were completely different across the two monthly samples. Consequently, attraction composition provides a particularly strong alternative explanation for the observed change in the aggregate emotional profile.

The episode is therefore classified as a **Weak contextual association**. The July heatwave clearly altered the tourism environment in Rome, but the available review evidence does not establish that it produced the detected emotional drift.

This case illustrates the distinction between identifying a real external event and demonstrating that the event explains an emotional change. External temporal alignment alone is insufficient when strong internal composition effects are present.

In [47]:
# ============================================================
# STEP 11M: MAP MARRAKECH AUGUST-SEPTEMBER 2024 CONTEXT
# ============================================================

event_id = "Marrakech_2024-08_2024-09"

mask = external_event_mapping["Event_ID"] == event_id

external_event_mapping.loc[
    mask, "Possible_External_Event"
] = (
    "First anniversary of the Al Haouz earthquake amid "
    "continuing regional recovery and tourism activity"
)

external_event_mapping.loc[
    mask, "Event_Date"
] = "8 September 2024"

external_event_mapping.loc[
    mask, "Event_Category"
] = "Post-disaster recovery / destination context"

external_event_mapping.loc[
    mask, "Review_Evidence"
] = (
    "Neutral decreased by 0.168 while Sadness increased by 0.152. "
    "Average rating remained almost unchanged at 3.28 versus 3.23. "
    "Review volume declined from 40 to 22, while attraction overlap "
    "remained relatively high at 0.75. September keywords included "
    "terms such as 'surprising' and 'scammed'."
)

external_event_mapping.loc[
    mask, "External_Source"
] = (
    "Reuters / Morocco tourism statistics"
)

external_event_mapping.loc[
    mask, "Evidence_Strength"
] = "Weak"

external_event_mapping.loc[
    mask, "Interpretation"
] = (
    "The September 2024 emotional transition occurred around the "
    "first anniversary of the September 2023 Al Haouz earthquake, "
    "while reconstruction and regional recovery were still ongoing. "
    "This provides a temporally and geographically relevant destination "
    "context. However, the review text does not directly connect the "
    "increase in Sadness to earthquake recovery, and average ratings "
    "remained almost unchanged. The decline in review volume and other "
    "visitor-experience factors may also have contributed. The external "
    "association is therefore classified as Weak and is not interpreted "
    "as causal."
)

display(
    external_event_mapping[
        external_event_mapping["Event_ID"] == event_id
    ]
)

,Event_ID,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Rating_Change,Attraction_Overlap,Before_Keywords,After_Keywords,Possible_External_Event,Event_Date,Event_Category,Review_Evidence,External_Source,Evidence_Strength,Interpretation
11,Marrakech_2024-08_2024-09,Marrakech,2024-08-01,2024-09-01,2.571831,Notable Drift,Neutral,Decrease,-0.167827,-0.047727,0.75,"beautiful, place, expensive, great, garden, av...","beautiful, totally, place, surprising, scammed...",First anniversary of the Al Haouz earthquake a...,8 September 2024,Post-disaster recovery / destination context,Neutral decreased by 0.168 while Sadness incre...,Reuters / Morocco tourism statistics,Weak,The September 2024 emotional transition occurr...


### Interpretation of the Marrakech August–September 2024 Episode

The Marrakech transition from August to September 2024 was identified as a **Notable Drift** episode with a Robust Z-score of **2.57**. The principal emotional change was a **decrease in Neutral emotion (-0.168)** accompanied by an **increase in Sadness (+0.152)**.

Unlike some other episodes, average rating remained almost unchanged, decreasing only from **3.28 to 3.23**. Review volume declined from 40 to 22 reviews, while attraction overlap remained relatively high at **0.75**.

External investigation identified a relevant destination context. September 2024 marked the **first anniversary of the 8 September 2023 Al Haouz earthquake**, which severely affected communities in the region surrounding Marrakech. Contemporary reporting around the anniversary indicated that reconstruction and recovery remained ongoing in affected communities.

At the same time, Morocco's tourism sector had experienced substantial recovery, with international arrivals continuing to grow during 2024. September therefore represented a period in which strong tourism activity coexisted with continuing post-earthquake recovery in the wider Marrakech region.

This context is temporally and geographically relevant to the detected emotional transition. However, the review-level evidence does not directly connect the observed increase in Sadness to the earthquake anniversary or reconstruction process. September keywords such as *scammed* instead suggest that ordinary visitor-experience issues may also have contributed to the emotional profile.

Furthermore, the average rating remained nearly unchanged, despite the measurable shift in the seven-dimensional emotion distribution.

The episode is therefore classified as a **Weak contextual association**. The post-earthquake recovery environment provides relevant external context, but the available evidence is insufficient to attribute the emotional drift directly to the earthquake anniversary or its continuing effects.

This case further demonstrates that external-event mapping is used for contextual interpretation rather than causal attribution.

### Validate all event mappings

In [48]:
# ============================================================
# STEP 12A: VALIDATE FINAL EXTERNAL EVENT MAPPING
# ============================================================

print("Total candidate episodes:", len(external_event_mapping))
print("\nEvidence-strength distribution:")
print(external_event_mapping["Evidence_Strength"].value_counts())

# Check that every candidate has been investigated
required_columns = [
    "Possible_External_Event",
    "Event_Date",
    "Event_Category",
    "External_Source",
    "Evidence_Strength",
    "Interpretation"
]

missing_mapping = external_event_mapping[
    external_event_mapping[required_columns]
    .replace("", pd.NA)
    .isna()
    .any(axis=1)
]

if missing_mapping.empty:
    print("\n✓ All 12 candidate episodes have been investigated.")
else:
    print("\n⚠ Some mappings are incomplete:")
    display(
        missing_mapping[
            ["Event_ID", "City", "Previous_Month", "Drift_Month"]
        ]
    )

Total candidate episodes: 12

Evidence-strength distribution:
Evidence_Strength
Moderate                                6
Weak                                    4
No identifiable external association    2
Name: count, dtype: int64

✓ All 12 candidate episodes have been investigated.


In [49]:
# ============================================================
# STEP 12B: FINAL EVENT-MAPPING SUMMARY
# ============================================================

final_event_summary = external_event_mapping[
    [
        "City",
        "Previous_Month",
        "Drift_Month",
        "Robust_Z",
        "Drift_Category",
        "Main_Emotion_Change",
        "Emotion_Direction",
        "Emotion_Change",
        "Possible_External_Event",
        "Event_Category",
        "Evidence_Strength"
    ]
].copy()

final_event_summary = final_event_summary.sort_values(
    "Robust_Z",
    ascending=False
).reset_index(drop=True)

display(final_event_summary)

,City,Previous_Month,Drift_Month,Robust_Z,Drift_Category,Main_Emotion_Change,Emotion_Direction,Emotion_Change,Possible_External_Event,Event_Category,Evidence_Strength
0,Miami,2020-11-01,2020-12-01,4.599706,Extreme Drift,Disgust,Decrease,-0.364703,Post-COVID reopening and expansion of tourism ...,COVID-19 tourism recovery / attraction reopening,Moderate
1,Marrakech,2024-11-01,2024-12-01,4.523351,Extreme Drift,Joy,Decrease,-0.266616,Ongoing animal-welfare concerns involving Barb...,Destination-specific animal-welfare issue,Moderate
2,Paris,2021-03-01,2021-04-01,3.365465,Notable Drift,Neutral,Increase,0.290143,COVID-19 lockdown and severe restrictions on t...,COVID-19 restrictions / tourism disruption,Moderate
3,Barcelona,2022-07-01,2022-08-01,3.195793,Notable Drift,Neutral,Increase,0.293166,No specific external event identified,No identifiable external association,No identifiable external association
4,Marrakech,2021-10-01,2021-11-01,3.127093,Notable Drift,Joy,Decrease,-0.201290,Progressive COVID-19 international flight rest...,COVID-19 travel restrictions / tourism disruption,Moderate
5,Cairo,2023-03-01,2023-04-01,2.850124,Notable Drift,Joy,Decrease,-0.258351,Ramadan 2023 and associated changes in daily o...,Religious period / tourism operating environment,Weak
6,Cairo,2020-08-01,2020-09-01,2.782004,Notable Drift,Neutral,Decrease,-0.307213,Resumption of cultural tourism and nationwide ...,COVID-19 reopening / cultural tourism recovery,Moderate
7,Bali,2022-11-01,2022-12-01,2.771473,Notable Drift,Joy,Increase,0.334752,Strong year-end increase in Bali tourism activ...,Tourism recovery / year-end tourism surge,Moderate
8,Dubai,2022-07-01,2022-08-01,2.761816,Notable Drift,Surprise,Increase,0.245576,No specific external event identified,No identifiable external association,No identifiable external association
9,Marrakech,2019-06-01,2019-07-01,2.628348,Notable Drift,Joy,Decrease,-0.260383,Transition to substantially hotter summer sigh...,Seasonal weather / tourism environment,Weak


In [52]:
# ============================================================
# STEP 12C: SAVE FINAL EXPANDED EVENT MAPPING
# ============================================================

output_path = results_dir / "Expanded_External_Event_Mapping.csv"
summary_path = results_dir / "Expanded_Event_Mapping_Summary.csv"

external_event_mapping.to_csv(
    output_path,
    index=False
)

final_event_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:")
print(" -", output_path)
print(" -", summary_path)

Saved:
 - C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews\results\Expanded_External_Event_Mapping.csv
 - C:\Users\suvar\OneDrive\ESILV\WebScrapping\TourismReviews\results\Expanded_Event_Mapping_Summary.csv


## Final Event-Mapping Summary

External-event investigation was performed for all **12 emotional-drift
episodes identified using the exploratory Robust Z > 2.5 threshold**. Importantly,
the candidate periods were selected from the emotional-drift analysis before
external events were investigated. This reduces the risk of selecting periods
because a known real-world event was already expected.

The external investigation produced three levels of contextual evidence:

- **Moderate association (6 episodes):** multiple internal indicators and
  independently identified external context showed meaningful temporal alignment.
- **Weak association (4 episodes):** a plausible and temporally relevant external
  context was identified, but internal evidence, sample composition, or other
  limitations prevented stronger interpretation.
- **No identifiable external association (2 episodes):** external investigation
  did not identify a sufficiently specific event capable of distinguishing the
  two comparison periods.

The two episodes classified as **Extreme Drift (Robust Z > 3.5)** remained the
principal findings of the study: Miami (November–December 2020) and Marrakech
(November–December 2024). The additional ten episodes were treated as exploratory
Notable Drift cases and provide supplementary evidence about how emotional
profiles can change under different tourism contexts.

External-event mapping was used for **contextual interpretation rather than
causal attribution**. Temporal coincidence alone was not considered sufficient
evidence that an external event caused a change in tourist emotions. Review
volume, attraction composition, ratings, emotion probabilities, review text,
keywords, and external evidence were considered together when assigning the
strength of each association.